In [40]:
!pip install mediapipe==0.10.13


In [41]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [42]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from mediapipe.python.solutions.hands import Hands, HandLandmark, HAND_CONNECTIONS
from mediapipe.python.solutions.pose import Pose, PoseLandmark

print("✅ 임포트 완료")


✅ 임포트 완료


In [43]:
# ──────────────────────────────────────────────
# Google Drive MP4 파일 ID 입력
# 170~180 프레임 구간을 이미지로 저장
# ──────────────────────────────────────────────

VIDEO_FILE_ID = "1qYz58DDcRQobH9yEwWuEw7x1a7YJ1SSe"
VIDEO_PATH    = "/content/input.mp4"
FRAME_DIR     = "/content/frames"

START_FRAME = 170
END_FRAME   = 180

os.makedirs(FRAME_DIR, exist_ok=True)

# Drive에서 다운로드
import urllib.request
url = f"https://drive.google.com/uc?export=download&id={VIDEO_FILE_ID}"
print("[다운로드 중...]")
urllib.request.urlretrieve(url, VIDEO_PATH)
print(f"✅ 다운로드 완료: {VIDEO_PATH}")

# 프레임 추출
cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
print(f"총 프레임 수: {total_frames}, FPS: {fps:.1f}")

saved_paths = []
cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)

for frame_idx in range(START_FRAME, END_FRAME + 1):
    ret, frame = cap.read()
    if not ret:
        print(f"프레임 {frame_idx} 읽기 실패")
        break
    save_path = os.path.join(FRAME_DIR, f"frame_{frame_idx:05d}.jpg")
    cv2.imwrite(save_path, frame)
    saved_paths.append(save_path)

cap.release()
print(f"\n✅ 추출 완료: {len(saved_paths)}장 → {FRAME_DIR}")


[다운로드 중...]
✅ 다운로드 완료: /content/input.mp4
총 프레임 수: 406, FPS: 59.6

✅ 추출 완료: 11장 → /content/frames


In [44]:
# ──────────────────────────────────────────────
# Pose: 전신에서 손목 위치 감지
# Hands: 손목 crop에서 손가락 좌표 추출
# ──────────────────────────────────────────────

pose_detector = Pose(
    static_image_mode=True,
    model_complexity=2,
    min_detection_confidence=0.3,
)

hands_detector = Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.1,
)

LANDMARK_NAMES = [lm.name for lm in HandLandmark]
print("✅ 모델 초기화 완료")


✅ 모델 초기화 완료


In [45]:
def extract_landmarks(image_path, pose_det, hand_det, padding=2.0):
    """
    1단계: Pose로 손목 위치 감지
    2단계: 손목 기준 crop
    3단계: Hands로 손가락 랜드마크 추출 (원본 픽셀 좌표로 저장)
    """
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        return None

    H, W = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    pose_result = pose_det.process(img_rgb)
    if not pose_result.pose_landmarks:
        return None

    lms      = pose_result.pose_landmarks.landmark
    all_rows = []

    for wrist_id in [PoseLandmark.LEFT_WRIST, PoseLandmark.RIGHT_WRIST]:
        wrist = lms[wrist_id]
        if wrist.visibility < 0.00001:
            continue

        cx, cy = int(wrist.x * W), int(wrist.y * H)
        pad    = int(H * 0.15 * padding)
        x1, y1 = max(0, cx - pad), max(0, cy - pad)
        x2, y2 = min(W, cx + pad), min(H, cy + pad)

        crop = img_bgr[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        crop_h, crop_w = crop.shape[:2]
        crop_rgb    = cv2.cvtColor(cv2.resize(crop, (256, 256)), cv2.COLOR_BGR2RGB)
        hand_result = hand_det.process(crop_rgb)
        if not hand_result.multi_hand_landmarks:
            continue

        side = "LEFT" if wrist_id == PoseLandmark.LEFT_WRIST else "RIGHT"

        for hand_idx, hand_lms in enumerate(hand_result.multi_hand_landmarks):
            handedness = hand_result.multi_handedness[hand_idx].classification[0].label
            for lm_idx, lm in enumerate(hand_lms.landmark):
                # crop 내 정규화 좌표 → 원본 픽셀 좌표 역변환
                px = lm.x * crop_w + x1
                py = lm.y * crop_h + y1
                all_rows.append({
                    "frame"      : Path(image_path).stem,
                    "wrist_side" : side,
                    "handedness" : handedness,
                    "landmark_id": lm_idx,
                    "landmark"   : LANDMARK_NAMES[lm_idx],
                    "x_px"       : px,
                    "y_px"       : py,
                    # crop 영역 저장 (시각화용)
                    "crop_x1"    : x1,
                    "crop_y1"    : y1,
                    "crop_x2"    : x2,
                    "crop_y2"    : y2,
                })

    return all_rows if all_rows else None


In [46]:
all_records = []
fail_list   = []

for path in saved_paths:
    records = extract_landmarks(path, pose_detector, hands_detector)
    if records is None:
        fail_list.append(Path(path).stem)
    else:
        all_records.extend(records)

print(f"✅ 완료 — 성공: {len(saved_paths) - len(fail_list)}, 실패: {len(fail_list)}")
if fail_list:
    print("실패 프레임:", fail_list)

df = pd.DataFrame(all_records)
print(f"총 랜드마크: {len(df)}")
df.head()


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


✅ 완료 — 성공: 0, 실패: 11
실패 프레임: ['frame_00170', 'frame_00171', 'frame_00172', 'frame_00173', 'frame_00174', 'frame_00175', 'frame_00176', 'frame_00177', 'frame_00178', 'frame_00179', 'frame_00180']
총 랜드마크: 0


""


In [52]:
# ──────────────────────────────────────────────
# 전체 406프레임 중 Hands가 감지되는 프레임 찾기
# ──────────────────────────────────────────────

cap = cv2.VideoCapture(VIDEO_PATH)
detected_frames = []

for frame_idx in range(406):
    ret, frame = cap.read()
    if not ret:
        break

    # 빠른 탐색용: 전체 이미지를 256x256으로 줄여서 Hands 시도
    small = cv2.resize(frame, (256, 256))
    rgb   = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
    result = hands_detector.process(rgb)

    if result.multi_hand_landmarks:
        detected_frames.append(frame_idx)

cap.release()
print(f"Hands 감지된 프레임: {detected_frames}")
print(f"총 {len(detected_frames)}개")


Hands 감지된 프레임: []
총 0개


In [47]:
# ──────────────────────────────────────────────
# 감지된 전 프레임을 한 화면에 grid로 표시
# 행: 프레임, 열: LEFT / RIGHT hand crop
# ──────────────────────────────────────────────

success_paths = [p for p in saved_paths if Path(p).stem not in fail_list]
n = len(success_paths)
print(f"시각화 대상: {n}장")

fig, axes = plt.subplots(n, 2, figsize=(12, 5 * n))

# n=1일 때 axes가 1D로 생성되는 문제 방지
if n == 1:
    axes = [axes]

for row_idx, img_path in enumerate(success_paths):
    img_bgr    = cv2.imread(img_path)
    frame_name = Path(img_path).stem
    frame_df   = df[df["frame"] == frame_name]

    for col_idx, side in enumerate(["LEFT", "RIGHT"]):
        ax      = axes[row_idx][col_idx]
        side_df = frame_df[frame_df["wrist_side"] == side]

        if side_df.empty:
            ax.set_title(f"{frame_name}\n{side} - 미감지", fontsize=8, color="red")
            ax.axis("off")
            continue

        x1 = int(side_df["crop_x1"].iloc[0])
        y1 = int(side_df["crop_y1"].iloc[0])
        x2 = int(side_df["crop_x2"].iloc[0])
        y2 = int(side_df["crop_y2"].iloc[0])
        crop_h = y2 - y1
        crop_w = x2 - x1

        crop     = img_bgr[y1:y2, x1:x2].copy()
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        lm_df    = side_df.set_index("landmark")

        # 연결선
        for connection in HAND_CONNECTIONS:
            a_name = LANDMARK_NAMES[connection[0]]
            b_name = LANDMARK_NAMES[connection[1]]
            if a_name in lm_df.index and b_name in lm_df.index:
                ax_ = int(lm_df.loc[a_name, "x_px"] - x1)
                ay_ = int(lm_df.loc[a_name, "y_px"] - y1)
                bx_ = int(lm_df.loc[b_name, "x_px"] - x1)
                by_ = int(lm_df.loc[b_name, "y_px"] - y1)
                cv2.line(crop_rgb, (ax_, ay_), (bx_, by_), (255, 255, 0), 2)

        # 랜드마크 점
        for _, row in lm_df.iterrows():
            px = int(row["x_px"] - x1)
            py = int(row["y_px"] - y1)
            if 0 <= px < crop_w and 0 <= py < crop_h:
                cv2.circle(crop_rgb, (px, py), 5, (0, 255, 0), -1)

        ax.imshow(crop_rgb)
        ax.set_title(f"{frame_name} | {side}", fontsize=8)
        ax.axis("off")

plt.tight_layout()
plt.show()


시각화 대상: 0장


ValueError: Number of rows must be a positive integer, not 0

<Figure size 1200x0 with 0 Axes>